# 11 - Model Interpretation & Decision Threshold Optimization

## 1. Objectives
- Extract and compare feature importance rankings across all three algorithms.
- Analyze the Precision-Recall trade-off across decision thresholds $[0.01, 0.99]$.
- Identify the optimal decision boundary for maximum F1-Score.

In [1]:
import sys
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import MODEL_PATH, RAW_DATA_PATH
from src.data.data_loader import load_and_split_data
from src.utils.plotting import plot_threshold_analysis, plot_feature_importance

X_train, X_test, y_train, y_test = load_and_split_data(RAW_DATA_PATH)
model_pipeline = joblib.load(MODEL_PATH)
print("Loaded production pipeline from:", MODEL_PATH)

2026-10-26 17:56:24 [INFO] data_loader (data_loader.py:42): Loading raw dataset from D:\Personal lone\data\raw\Bank_Personal_Loan_Modelling.csv


2026-10-26 17:56:24 [INFO] data_loader (data_loader.py:44): Loaded raw dataset with shape (5000, 14)


2026-10-26 17:56:24 [INFO] data_validation (validation.py:88): Raw data validation passed successfully.


2026-10-26 17:56:24 [INFO] data_loader (data_loader.py:78): Performing stratified train-test split (test_size=0.20, seed=42)


2026-10-26 17:56:25 [INFO] data_loader (data_loader.py:92): Train shape: (4000, 13), Test shape: (1000, 13)


2026-10-26 17:56:25 [INFO] data_loader (data_loader.py:93): Train target balance: {0: np.int64(3616), 1: np.int64(384)} (9.6% positive)


2026-10-26 17:56:25 [INFO] data_loader (data_loader.py:98): Test target balance: {0: np.int64(904), 1: np.int64(96)} (9.6% positive)


Loaded production pipeline from: D:\Personal lone\artifacts\models\bank_personal_loan_model.joblib


## 2. Feature Importance Analysis

In [2]:
classifier = model_pipeline.named_steps["classifier"]
fe = model_pipeline.named_steps["preprocessor"].named_steps["feature_engineer"]
feature_names = fe.get_feature_names_out()

if hasattr(classifier, "feature_importances_"):
    importances = classifier.feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
    fig = plot_feature_importance(imp_df, top_n=12, title="Champion Model Feature Importances")
    plt.show()
    display(imp_df.head(10))

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1640\3303195169.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,importance
2,Income,24.904840
5,Education,20.453315
4,CCAvg,10.303036
3,Family,9.518781
11,Income_Per_Family,7.141229
12,CCAvg_to_Income_Ratio,6.745763
9,Online,3.413325
0,Age,3.304269
16,Experience_to_Age_Ratio,2.386843
1,Experience,2.352251


## 3. Decision Threshold Calibration Curve

In [3]:
y_probs = model_pipeline.predict_proba(X_test)[:, 1]
fig, optimal_threshold = plot_threshold_analysis(y_test.values, y_probs)
plt.show()

print(f"Optimal F1 Decision Threshold: {optimal_threshold:.3f}")

Optimal F1 Decision Threshold: 0.634


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1640\4197113589.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Default (0.50) vs Calibrated Threshold Comparison

In [4]:
from src.models.evaluation import evaluate_model

preds_default = (y_probs >= 0.50).astype(int)
preds_calibrated = (y_probs >= optimal_threshold).astype(int)

m_default = evaluate_model(y_test, preds_default, y_probs, model_name="Default (0.50)")
m_calibrated = evaluate_model(y_test, preds_calibrated, y_probs, model_name=f"Calibrated ({optimal_threshold:.2f})")

comp_thresh = pd.DataFrame([m_default, m_calibrated])
display(comp_thresh[["model_name", "accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]])

,model_name,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Default (0.50),0.992,0.9490,0.9688,0.9588,0.9995,0.9955
1,Calibrated (0.63),0.994,0.9688,0.9688,0.9688,0.9995,0.9955
